# Classificador de Qualidade de Lances de Xadrez

**Disciplina:** Paradigmas de Aprendizagem de Máquina  

---

## 1. Introdução

### Problema

Jogadores de xadrez de nível intermediário (rating 1200–1500 Chess.com, equivalente a 1400–1700 no Lichess) cometem erros frequentes durante o meio-jogo: peças penduradas, trocas desfavoráveis, descuido com a segurança do rei. Este projeto constrói um **classificador supervisionado** que distingue lances **bons** de **ruins** com base em features posicionais.

### Objetivo

Treinar e comparar dois modelos de classificação — **Decision Tree** (obrigatório, interpretável) e **Random Forest** (ensemble, para comparação) — demonstrando o pipeline completo de ML: coleta, rotulagem, engenharia de features, treino com tuning, avaliação e interpretação no domínio.

### Versões do pipeline

| Versão | Features | Descrição |
|--------|----------|-----------|
| **V1** | 33 posicionais | Material, mobilidade, segurança do rei, estrutura de peões, controle do centro, características do lance, contexto |
| **V2** | 52 (33 + 19 táticas) | V1 + peças indefesas, capturas com ganho, cravadas, rei avançado, tensão |
| **V3** | 67 (52 + 15 look-ahead) | V2 + deltas antes/depois, resposta do adversário, Static Exchange Evaluation (SEE) |

O notebook é controlado por duas flags na célula de configuração:
- **`VERSION`** (1, 2 ou 3): seleciona qual conjunto de features, modelos e avaliação usar
- **`RERUN_PIPELINE`** (True/False): re-executa o pipeline do zero ou usa dados pré-computados

### Abordagem resumida

| Etapa | Método |
|-------|--------|
| Dados | Partidas reais do Lichess (CC0), filtradas por rating e tempo |
| Rotulagem | Avaliação posicional via Stockfish (depth 15): bom (≥ −50 cp), ruim (≤ −150 cp) |
| Features | V1: 33 posicionais · V2: 52 (+ táticas) · V3: 67 (+ look-ahead) — `python-chess` |
| Modelos | Decision Tree + Random Forest, com `class_weight="balanced"` |
| Métricas | Foco em recall e F1 da classe "ruim" — detectar erros é o objetivo |

In [ ]:
import json
import os
import random
import sys
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import learning_curve, train_test_split
from sklearn.tree import export_text, plot_tree

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"font.size": 11, "figure.facecolor": "white"})
sns.set_style("whitegrid")

PROJECT_ROOT = Path("..").resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = Path("data")

print(f"Diretório de trabalho: {os.getcwd()}")
print("Setup OK")

In [ ]:
VERSION = 3            # 1 = posicional (33), 2 = + tática (52), 3 = + look-ahead (67)
RERUN_PIPELINE = False  # True = re-executa todo o pipeline, False = usa dados pré-computados

_v_suffixes = {1: "", 2: "_v2", 3: "_v3"}
_v_suffix = _v_suffixes[VERSION]
FEATURES_CSV = DATA_DIR / "features" / f"features{_v_suffix}.csv"
MODELS_DIR = DATA_DIR / f"models{_v_suffix}"
EVAL_DIR = DATA_DIR / f"evaluation{_v_suffix}"

_ver_labels = {
    1: "V1 — posicional (33 features)",
    2: "V2 — posicional + tática (52 features)",
    3: "V3 — posicional + tática + look-ahead (67 features)",
}
print(f"VERSION        = {VERSION}  ({_ver_labels[VERSION]})")
print(f"RERUN_PIPELINE = {RERUN_PIPELINE}")
print(f"Features CSV   : {FEATURES_CSV}")
print(f"Modelos        : {MODELS_DIR}")
print(f"Avaliação      : {EVAL_DIR}")
print()
if RERUN_PIPELINE:
    print("⚠️  O pipeline completo será re-executado (pode levar >3 horas).")
else:
    print("Usando dados pré-computados.")

---

## 2. Coleta e Descrição dos Dados

### Fonte

Os dados vêm da **Lichess open database** ([database.lichess.org](https://database.lichess.org)), sob licença **CC0** (domínio público).

- **Ficheiro:** `lichess_db_standard_rated_2015-01.pgn.zst` (~272 MiB comprimido)
- **Formato:** PGN comprimido com Zstandard, processado em streaming (memória constante)

### Filtros aplicados

| Filtro | Critério | Justificativa |
|--------|----------|---------------|
| Rating | Ambos 1400–1700 (Lichess) | Faixa-alvo equivalente a 1200–1500 Chess.com |
| Tempo | Blitz/Rapid (3–10 min) | Jogos sérios mas com pressão temporal |
| Término | Normal (mate/resignação) | Evitar jogos decididos por timeout |
| Variante | Standard | Sem Chess960 etc. |
| Fase | Lances 8 a 40 | Meio-jogo (exclui abertura teórica e finais simplificados) |
| Amostragem | 10% das partidas válidas | Controle de volume, seed=42 |

In [ ]:
if RERUN_PIPELINE:
    from download_pgn import download

    PGN_URL = "https://database.lichess.org/standard/lichess_db_standard_rated_2015-01.pgn.zst"
    PGN_PATH = DATA_DIR / "raw" / "lichess_db_standard_rated_2015-01.pgn.zst"

    if not PGN_PATH.exists():
        print("Baixando PGN do Lichess (~272 MiB)...")
        download(PGN_URL, PGN_PATH)
    else:
        print(f"PGN já existe: {PGN_PATH}")
else:
    print("⏭️  Download do PGN pulado (usando dados pré-computados).")

In [ ]:
if RERUN_PIPELINE:
    from filter_games import filter_and_sample, print_stats

    print("Filtrando e amostrando partidas...")
    print("Critérios: Elo 1400–1700, Blitz/Rapid, Normal, lances 8–40")
    print("Amostragem: 10%, seed=42, max 3000 partidas\n")

    stats = filter_and_sample(
        pgn_path=DATA_DIR / "raw" / "lichess_db_standard_rated_2015-01.pgn.zst",
        output_path=DATA_DIR / "filtered" / "moves_filtered.csv",
        sample_rate=0.10,
        seed=42,
        max_games=3000,
    )
    print_stats(stats)
else:
    print("⏭️  Filtragem pulada (usando CSV pré-computado).")

In [ ]:
df_filtered = pd.read_csv(DATA_DIR / "filtered" / "moves_filtered.csv")

n_games = df_filtered["game_site"].nunique()
n_moves = len(df_filtered)
avg_moves = n_moves / n_games

print(f"{'='*50}")
print(f"  DATASET FILTRADO")
print(f"{'='*50}")
print(f"  Partidas       : {n_games:,}")
print(f"  Lances (total) : {n_moves:,}")
print(f"  Lances/partida : {avg_moves:.1f}")
print(f"  Colunas        : {list(df_filtered.columns)}")
print()
df_filtered.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, col, title, color in [
    (axes[0], "white_elo", "Rating das Brancas", "#4C72B0"),
    (axes[1], "black_elo", "Rating das Pretas", "#DD8452"),
]:
    elos = df_filtered.groupby("game_site")[col].first()
    ax.hist(elos, bins=30, color=color, edgecolor="white", alpha=0.85)
    ax.set_xlabel("Elo")
    ax.set_ylabel("Partidas")
    ax.set_title(title)
    ax.axvline(elos.mean(), color="red", linestyle="--",
               label=f"Média: {elos.mean():.0f}")
    ax.legend()

plt.suptitle("Distribuição de Ratings no Dataset Filtrado", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## 3. Rotulagem via Stockfish

### Conceito: centipawn loss (delta)

Para cada lance, a qualidade é medida pela **variação da avaliação** que ele causa:

$$\delta = \text{eval\_depois} - \text{eval\_antes}$$

onde ambas as avaliações são do ponto de vista do jogador que fez o lance. Um delta muito negativo significa que o lance **piorou** a posição.

### Limiares de classificação

| Classe | Condição | Significado | Analogia |
|--------|----------|-------------|----------|
| **Bom** | δ ≥ −50 cp | Perda ≤ 0.5 peão | Lance aceitável |
| **Descartado** | −150 < δ < −50 | Imprecisão intermediária | Zona cinzenta — ambíguo |
| **Ruim** | δ ≤ −150 cp | Perda ≥ 1.5 peão | Erro claro |

A **zona cinzenta** é descartada para reduzir ruído: lances que perdem entre 0.5 e 1.5 peão são ambíguos demais para rotular com confiança.

### Configuração do Stockfish

| Parâmetro | Valor | Justificativa |
|-----------|-------|---------------|
| Engine | Stockfish 18 | State-of-the-art, open source |
| Profundidade | 15 | Detecta táticas de 3–4 lances; equilíbrio custo/qualidade |
| Workers | 6 paralelos | Cada um com 1 thread + 64 MB hash |
| Caching | eval_after reutilizado como eval_before do lance seguinte | Corta avaliações quase pela metade |
| Tempo total | ~174 minutos | Apple M4 |

In [ ]:
if RERUN_PIPELINE:
    from label_moves import run as label_run

    print("Rotulando lances com Stockfish (depth 15, 6 workers)...")
    print("⚠️  Este passo leva ~174 minutos.\n")

    label_run(num_workers=6)
else:
    print("⏭️  Rotulagem pulada (usando CSVs pré-computados).")
    print(f"   → {DATA_DIR / 'labeled' / 'moves_all_scored.csv'}")
    print(f"   → {DATA_DIR / 'labeled' / 'moves_labeled.csv'}")

In [ ]:
df_scored = pd.read_csv(DATA_DIR / "labeled" / "moves_all_scored.csv")
df_labeled = pd.read_csv(DATA_DIR / "labeled" / "moves_labeled.csv")

n_bom = (df_labeled["label"] == "bom").sum()
n_ruim = (df_labeled["label"] == "ruim").sum()
n_desc = len(df_scored) - len(df_labeled)

print(f"{'='*50}")
print(f"  ROTULAGEM")
print(f"{'='*50}")
print(f"  Lances avaliados      : {len(df_scored):,}")
print(f"  Bom (δ ≥ −50 cp)      : {n_bom:,} ({n_bom/len(df_scored)*100:.1f}%)")
print(f"  Descartado (cinzenta) : {n_desc:,} ({n_desc/len(df_scored)*100:.1f}%)")
print(f"  Ruim (δ ≤ −150 cp)    : {n_ruim:,} ({n_ruim/len(df_scored)*100:.1f}%)")
print(f"  ──────────────────────")
print(f"  Dataset final         : {len(df_labeled):,} (bom + ruim)")
print(f"  Ratio bom:ruim        : {n_bom/n_ruim:.2f}:1")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ax = axes[0]
deltas = df_scored["delta_cp"].clip(-2000, 2000)
ax.hist(deltas, bins=80, color="#4C72B0", edgecolor="white", alpha=0.85)
ax.axvline(-50, color="green", linestyle="--", linewidth=1.5, label="Limiar bom (−50)")
ax.axvline(-150, color="red", linestyle="--", linewidth=1.5, label="Limiar ruim (−150)")
ax.set_xlabel("Delta (centipawns)")
ax.set_ylabel("Lances")
ax.set_title("Distribuição de Delta CP")
ax.legend(fontsize=9)

ax = axes[1]
counts = df_scored["label"].value_counts()
colors_map = {"bom": "#55A868", "ruim": "#C44E52", "descartado": "#8C8C8C"}
order = ["bom", "descartado", "ruim"]
bars = ax.bar(order, [counts.get(c, 0) for c in order],
              color=[colors_map.get(c, "gray") for c in order], edgecolor="white")
for bar, c in zip(bars, order):
    val = counts.get(c, 0)
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f"{val:,}\n({val/len(df_scored)*100:.1f}%)", ha="center", fontsize=10)
ax.set_ylabel("Lances")
ax.set_title("Distribuição por Classe")

ax = axes[2]
bom_delta = df_labeled[df_labeled["label"] == "bom"]["delta_cp"].clip(-3000, 3000)
ruim_delta = df_labeled[df_labeled["label"] == "ruim"]["delta_cp"].clip(-3000, 3000)
bp = ax.boxplot([bom_delta, ruim_delta], labels=["Bom", "Ruim"],
                patch_artist=True, widths=0.5)
bp["boxes"][0].set_facecolor("#55A868")
bp["boxes"][1].set_facecolor("#C44E52")
ax.set_ylabel("Delta (centipawns)")
ax.set_title("Delta CP por Classe")

plt.suptitle("Análise da Rotulagem", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## 4. Engenharia de Features

### Features V1 — Posicionais (33)

| Grupo | Qtd. | Features | Extração |
|-------|------|----------|----------|
| **Material** | 11 | Contagem por tipo/cor + diferença ponderada | `board.pieces()` |
| **Mobilidade** | 3 | Lances legais (jogador/adversário/diferença) | `board.legal_moves` |
| **Segurança do rei** | 4 | Roque, direito a roque, escudo de peões | Posição do rei + peões adjacentes |
| **Estrutura de peões** | 4 | Dobrados, isolados, passados | Análise de colunas |
| **Controle do centro** | 3 | Ataques e ocupação de d4/d5/e4/e5 | `board.is_attacked_by()` |
| **Características do lance** | 6 | Captura, xeque, promoção, peça, destino | `board.is_capture()` etc. |
| **Contexto** | 2 | Número do lance, cor | Metadados da partida |

### Features V2 — Táticas (+19, total 52)

| Grupo | Qtd. | Features | Extração |
|-------|------|----------|----------|
| **Peças indefesas** | 5 | Hanging pieces/value (jogador/adversário), min attacker vs piece | `board.attackers()` + `board.is_attacked_by()` |
| **Capturas com ganho** | 4 | Ameaças contra jogador/adversário, valor máximo de ameaça | Análise de atacantes vs valor da peça |
| **Cravadas** | 2 | Peças cravadas (jogador/adversário) | Detecção de pins via raios de ataque |
| **Rei avançado** | 4 | Atacantes do rei, colunas abertas, casas de fuga | Análise da zona do rei |
| **Tensão** | 4 | Ataques totais, casas disputadas, peças sem defesa | Contagem de ataques e casas contestadas |

### Features V3 — Look-ahead (+15, total 67)

| Grupo | Qtd. | Features | Extração |
|-------|------|----------|----------|
| **Deltas (antes→depois)** | 8 | Δ hanging, Δ tensão, Δ mobilidade, Δ ameaças, Δ atacantes do rei | `board.push(move)` + recálculo |
| **Resposta do adversário** | 4 | Melhor captura disponível, pode dar xeque, capturas boas, criou peça indefesa | Análise do tabuleiro pós-lance |
| **SEE** | 3 | SEE do lance, pior SEE contra jogador, é captura perdedora | Static Exchange Evaluation (swap algorithm) |

**Evolução conceitual:**

| Versão | Pergunta que responde | Analogia |
|--------|----------------------|----------|
| V1 | "Como é a posição?" | Exame físico geral |
| V2 | "Que perigos existem?" | Exames de diagnóstico |
| **V3** | **"O lance melhorou ou piorou a posição?"** | **Comparar antes e depois do tratamento** |

**Decisões de design:**
- Nenhum one-hot de casas individuais (64 casas = esparsidade sem ganho para árvores)
- Todas numéricas/inteiras, compatíveis diretamente com scikit-learn
- Sem normalização necessária para modelos baseados em árvore

In [ ]:
if RERUN_PIPELINE:
    from extract_features import run as extract_run

    _n_feat = {1: 33, 2: 52, 3: 67}
    print(f"Extraindo {_n_feat[VERSION]} features V{VERSION} de 109,290 lances (6 workers)...")
    extract_run(num_workers=6, batch_size=1000, v2=(VERSION >= 2), v3=(VERSION >= 3))
else:
    print(f"⏭️  Extração de features pulada (usando CSV pré-computado).")
    print(f"   → {FEATURES_CSV}")

In [ ]:
df_features = pd.read_csv(FEATURES_CSV)
feature_cols = [c for c in df_features.columns if c != "label"]

print(f"{'='*50}")
print(f"  FEATURES — V{VERSION}")
print(f"{'='*50}")
print(f"  Fonte       : {FEATURES_CSV}")
print(f"  Linhas      : {len(df_features):,}")
print(f"  Features    : {len(feature_cols)}")
print(f"  Valores nulos: {df_features[feature_cols].isnull().sum().sum()}")
print(f"\nDistribuição do label:")
print(df_features["label"].value_counts().to_string())
print(f"\nEstatísticas descritivas:")
df_features[feature_cols].describe().round(2)

In [ ]:
FEATURE_TRANSLATION = {
    # V1 — posicionais
    "white_pawns": "Peões brancos", "white_knights": "Cavalos brancos",
    "white_bishops": "Bispos brancos", "white_rooks": "Torres brancas",
    "white_queens": "Damas brancas", "black_pawns": "Peões pretos",
    "black_knights": "Cavalos pretos", "black_bishops": "Bispos pretos",
    "black_rooks": "Torres pretas", "black_queens": "Damas pretas",
    "material_diff": "Dif. material", "legal_moves_player": "Legais (jogador)",
    "legal_moves_opponent": "Legais (adversário)", "mobility_diff": "Dif. mobilidade",
    "player_castled": "Rocou", "opponent_castled": "Adv. rocou",
    "player_can_castle": "Pode rocar", "king_pawn_shield": "Escudo rei",
    "player_doubled_pawns": "Peões dobrados", "player_isolated_pawns": "Peões isolados",
    "player_passed_pawns": "Passados (jog.)", "opponent_passed_pawns": "Passados (adv.)",
    "player_center_control": "Centro (jog.)", "opponent_center_control": "Centro (adv.)",
    "player_center_occupation": "Ocupação centro", "is_capture": "É captura",
    "is_check": "Dá xeque", "is_promotion": "É promoção",
    "moved_piece": "Peça movida", "move_to_center": "Move p/ centro",
    "move_to_extended_center": "Move p/ centro exp.",
    "move_number": "Nº do lance", "is_white": "Joga de brancas",
    # V2 — táticas
    "hanging_pieces_player": "Indefesas (jog.)",
    "hanging_pieces_opponent": "Indefesas (adv.)",
    "hanging_value_player": "Val. indefeso (jog.)",
    "hanging_value_opponent": "Val. indefeso (adv.)",
    "min_attacker_vs_piece_player": "Menor atac. vs peça",
    "threats_against_player": "Ameaças (jog.)",
    "threats_against_opponent": "Ameaças (adv.)",
    "max_threat_value_player": "Maior ameaça (jog.)",
    "max_threat_value_opponent": "Maior ameaça (adv.)",
    "pinned_pieces_player": "Cravadas (jog.)",
    "pinned_pieces_opponent": "Cravadas (adv.)",
    "king_attackers_player": "Atacantes rei (jog.)",
    "king_attackers_opponent": "Atacantes rei (adv.)",
    "king_open_files_player": "Col. abertas rei",
    "king_escape_squares_player": "Fuga do rei",
    "total_attacks_player": "Ataques totais (jog.)",
    "total_attacks_opponent": "Ataques totais (adv.)",
    "contested_squares": "Casas disputadas",
    "undefended_pieces_player": "Sem defesa (jog.)",
    # V3 — look-ahead
    "delta_hanging_player": "Δ indefesas (jog.)",
    "delta_hanging_opponent": "Δ indefesas (adv.)",
    "delta_hanging_value_player": "Δ val. indefeso (jog.)",
    "delta_threats_against_player": "Δ ameaças (jog.)",
    "delta_mobility_player": "Δ mobilidade (jog.)",
    "delta_mobility_opponent": "Δ mobilidade (adv.)",
    "delta_contested_squares": "Δ casas disputadas",
    "delta_king_attackers_player": "Δ atacantes rei (jog.)",
    "opponent_best_capture_value": "Melhor captura adv.",
    "opponent_can_check": "Adv. pode dar xeque",
    "opponent_num_good_captures": "Nº capturas boas adv.",
    "created_hanging_self": "Criou peça indefesa",
    "see_of_move": "SEE do lance",
    "worst_see_against_player": "Pior SEE contra jog.",
    "is_losing_capture": "É captura perdedora",
}

corr = df_features[feature_cols].corr()
labels_pt = [FEATURE_TRANSLATION.get(c, c) for c in feature_cols]

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, cmap="RdBu_r", center=0,
    xticklabels=labels_pt, yticklabels=labels_pt,
    annot=False, square=True, linewidths=0.5,
    cbar_kws={"shrink": 0.7, "label": "Correlação de Pearson"}, ax=ax,
)
ax.set_title("Mapa de Correlação entre Features", fontsize=15, pad=15)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

high_corr = []
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        r = abs(corr.iloc[i, j])
        if r > 0.7:
            high_corr.append((feature_cols[i], feature_cols[j], corr.iloc[i, j]))

if high_corr:
    print("Pares com |correlação| > 0.7:")
    for a, b, r in sorted(high_corr, key=lambda x: -abs(x[2])):
        print(f"  {FEATURE_TRANSLATION.get(a, a):25s}  ↔  {FEATURE_TRANSLATION.get(b, b):25s}  r = {r:+.3f}")
else:
    print("Nenhum par de features com |correlação| > 0.7.")

---

## 5. Treino dos Modelos

### Split dos dados

O dataset foi dividido em **70/15/15** (treino/validação/teste) com **estratificação** para preservar a proporção bom/ruim em cada split. O conjunto de teste só é usado **uma vez**, na avaliação final.

### Desbalanceamento

A classe "ruim" representa ~15.6% do dataset (ratio **5.4:1**). Para mitigar o viés:
- `class_weight="balanced"` em ambos os modelos — ajusta o peso da loss proporcionalmente à frequência de cada classe

### Modelos

| Modelo | Por quê | Hiperparâmetros tunados |
|--------|---------|------------------------|
| **Decision Tree** | Obrigatório. Regras interpretáveis para o domínio. | `max_depth` ∈ {3,5,7,10,15,∅}, `min_samples_leaf` ∈ {1,5,10,20}, `criterion` ∈ {gini, entropy} |
| **Random Forest** | Ensemble de árvores — reduz variância. | `n_estimators` ∈ {50,100,200}, `max_depth` ∈ {5,10,15,∅}, `min_samples_leaf` ∈ {1,5,10} |

### Tuning

Ambos via `GridSearchCV` com **5 folds** e métrica **F1** da classe "ruim".

In [ ]:
X = df_features.drop(columns=["label"])
y = (df_features["label"] == "ruim").astype(int)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, stratify=y_temp, random_state=RANDOM_SEED
)

print(f"{'='*50}")
print(f"  SPLIT DOS DADOS")
print(f"{'='*50}")
for name, Xs, ys in [("Treino", X_train, y_train), ("Validação", X_val, y_val), ("Teste", X_test, y_test)]:
    n = len(ys)
    n_ruim = int(ys.sum())
    print(f"  {name:10s}: {n:>6,} amostras  (bom={n - n_ruim:,}, ruim={n_ruim:,}, {n_ruim/n*100:.1f}% ruim)")

In [ ]:
if RERUN_PIPELINE:
    from train_models import run as train_run

    print(f"Treinando modelos V{VERSION} (GridSearchCV, 5 folds)...")
    print("⚠️  Pode levar alguns minutos.\n")
    train_run(v2=(VERSION >= 2), v3=(VERSION >= 3))
else:
    print(f"⏭️  Treino pulado (usando modelos pré-treinados de {MODELS_DIR}).")

with open(MODELS_DIR / "feature_names.json") as f:
    feature_names = json.load(f)

dt = joblib.load(MODELS_DIR / "decision_tree.joblib")
rf = joblib.load(MODELS_DIR / "random_forest.joblib")

print(f"\n{'='*55}")
print(f"  DECISION TREE V{VERSION} — Melhores hiperparâmetros")
print(f"{'='*55}")
print(f"  criterion        : {dt.criterion}")
print(f"  max_depth        : {dt.max_depth}")
print(f"  min_samples_leaf : {dt.min_samples_leaf}")
print(f"  class_weight     : {dt.class_weight}")

print(f"\n{'='*55}")
print(f"  RANDOM FOREST V{VERSION} — Melhores hiperparâmetros")
print(f"{'='*55}")
print(f"  n_estimators     : {rf.n_estimators}")
print(f"  max_depth        : {rf.max_depth}")
print(f"  min_samples_leaf : {rf.min_samples_leaf}")
print(f"  class_weight     : {rf.class_weight}")

---

## 6. Avaliação

A avaliação é feita no **conjunto de teste** (16,394 lances) — nunca usado no treino nem no tuning.

### Métrica principal: F1 da classe "ruim"

A classe "ruim" é a mais importante: o objetivo é **detectar erros**. Um modelo que classifica tudo como "bom" teria ~84% de accuracy mas seria inútil.

| Métrica | O que mede | Por que importa |
|---------|------------|------------------|
| **Accuracy** | Acertos totais | Visão geral (enviesada pelo desbalanceamento) |
| **Recall (ruim)** | % de erros reais detectados | Não deixar erros escaparem |
| **Precision (ruim)** | % de alarmes que são reais | Evitar alarmes falsos |
| **F1 (ruim)** | Equilíbrio precision/recall | Métrica-chave |
| **ROC-AUC** | Capacidade discriminativa geral | Independente do threshold |

In [ ]:
y_pred_dt = dt.predict(X_test)
y_pred_rf = rf.predict(X_test)

print("=" * 60)
print("  DECISION TREE — Classification Report")
print("=" * 60)
print(classification_report(y_test, y_pred_dt, target_names=["bom", "ruim"]))

print("=" * 60)
print("  RANDOM FOREST — Classification Report")
print("=" * 60)
print(classification_report(y_test, y_pred_rf, target_names=["bom", "ruim"]))

In [ ]:
results = []
for name, model, y_pred in [
    ("Decision Tree", dt, y_pred_dt),
    ("Random Forest", rf, y_pred_rf),
]:
    y_proba = model.predict_proba(X_test)[:, 1]
    results.append({
        "Modelo": name,
        "Accuracy": f"{accuracy_score(y_test, y_pred):.4f}",
        "F1 (bom)": f"{f1_score(y_test, y_pred, pos_label=0):.4f}",
        "F1 (ruim)": f"{f1_score(y_test, y_pred, pos_label=1):.4f}",
        "Recall (ruim)": f"{(y_pred[y_test == 1] == 1).mean():.4f}",
        "Precision (ruim)": f"{(y_test[y_pred == 1] == 1).mean():.4f}",
        "ROC-AUC": f"{roc_auc_score(y_test, y_proba):.4f}",
    })

df_results = pd.DataFrame(results)
print("Tabela comparativa — Conjunto de teste:\n")
df_results

### 6.1 Matrizes de Confusão

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, model, name in [
    (axes[0], dt, "Decision Tree"),
    (axes[1], rf, "Random Forest"),
]:
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test,
        display_labels=["bom", "ruim"],
        cmap="Blues", ax=ax, values_format="d",
    )
    ax.set_title(f"Matriz de Confusão — {name}", fontsize=13)

plt.tight_layout()
plt.show()

### 6.2 Feature Importance

Quais features os modelos consideram mais importantes para distinguir lances bons de ruins?

In [ ]:
def plot_fi(model, feature_names, title, color, ax, top_n=15):
    imp = model.feature_importances_
    idx = np.argsort(imp)[::-1][:top_n]
    labels = [FEATURE_TRANSLATION.get(feature_names[i], feature_names[i]) for i in idx]
    values = imp[idx]
    bars = ax.barh(range(top_n), values, color=color)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel("Importância (Gini)")
    ax.set_title(title, fontsize=13)
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f"{val:.4f}", va="center", fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
plot_fi(dt, feature_names, "Top 15 Features — Decision Tree", "#4C72B0", axes[0])
plot_fi(rf, feature_names, "Top 15 Features — Random Forest", "#DD8452", axes[1])
plt.tight_layout()
plt.show()

### 6.3 Curvas ROC e Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
RocCurveDisplay.from_estimator(dt, X_test, y_test, ax=ax, name="Decision Tree")
RocCurveDisplay.from_estimator(rf, X_test, y_test, ax=ax, name="Random Forest")
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Aleatório")
ax.set_title("Curva ROC — Classe 'ruim'", fontsize=13)
ax.legend()

ax = axes[1]
PrecisionRecallDisplay.from_estimator(dt, X_test, y_test, ax=ax, name="Decision Tree")
PrecisionRecallDisplay.from_estimator(rf, X_test, y_test, ax=ax, name="Random Forest")
prevalence = y_test.mean()
ax.axhline(y=prevalence, color="k", linestyle="--", alpha=0.5,
           label=f"Prevalência ({prevalence:.3f})")
ax.set_title("Curva Precision-Recall — Classe 'ruim'", fontsize=13)
ax.legend()

plt.tight_layout()
plt.show()

### 6.4 Comparação Geral de Modelos

In [ ]:
metrics_data = {}
for name, model in [("Decision Tree", dt), ("Random Forest", rf)]:
    yp = model.predict(X_test)
    yproba = model.predict_proba(X_test)[:, 1]
    metrics_data[name] = {
        "Accuracy": accuracy_score(y_test, yp),
        "F1 (bom)": f1_score(y_test, yp, pos_label=0),
        "F1 (ruim)": f1_score(y_test, yp, pos_label=1),
        "Recall (ruim)": (yp[y_test == 1] == 1).mean(),
        "ROC-AUC": roc_auc_score(y_test, yproba),
    }

metric_names = list(next(iter(metrics_data.values())).keys())
x = np.arange(len(metric_names))
width = 0.30

fig, ax = plt.subplots(figsize=(11, 5))
for i, (model_name, vals) in enumerate(metrics_data.items()):
    values = [vals[m] for m in metric_names]
    bars = ax.bar(x + i * width, values, width, label=model_name,
                  color=["#4C72B0", "#DD8452"][i])
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.012,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x + width / 2)
ax.set_xticklabels(metric_names)
ax.set_ylim(0, 1.0)
ax.set_ylabel("Score")
ax.set_title("Comparação de Modelos — Conjunto de Teste", fontsize=14)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### 6.5 Curvas de Aprendizado

As curvas de aprendizado mostram como o F1-score evolui com o tamanho do conjunto de treino. Servem para responder: "precisamos de mais dados?" e "há overfitting?"

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
train_sizes_frac = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]

for ax, model, name in [
    (axes[0], dt, "Decision Tree"),
    (axes[1], rf, "Random Forest"),
]:
    sizes, train_scores, val_scores = learning_curve(
        model, X_train, y_train, cv=5,
        train_sizes=train_sizes_frac,
        scoring="f1", n_jobs=-1,
    )

    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)

    ax.fill_between(sizes, train_mean - train_std, train_mean + train_std,
                    alpha=0.1, color="#4C72B0")
    ax.fill_between(sizes, val_mean - val_std, val_mean + val_std,
                    alpha=0.1, color="#DD8452")
    ax.plot(sizes, train_mean, "o-", color="#4C72B0", label="Treino")
    ax.plot(sizes, val_mean, "o-", color="#DD8452", label="Validação (CV)")
    ax.set_xlabel("Tamanho do treino")
    ax.set_ylabel("F1-score")
    ax.set_title(f"Curva de Aprendizado — {name}", fontsize=13)
    ax.legend(loc="lower right")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 6.6 Diagnóstico: Por que o F1-ruim é baixo?

Antes de avançar para a interpretação, é importante entender **por que** o classificador tem dificuldade em detectar lances ruins. A resposta está na **capacidade discriminativa das features**.

#### Hipóteses descartadas

| Hipótese | Evidência contra |
|----------|-----------------|
| "Precisamos de mais dados" | Learning curves estáveis — validação plana |
| "O modelo é fraco" | RF e DT concordam; o sinal não existe nas features |
| "O desbalanceamento não foi tratado" | `class_weight="balanced"` já ativo |
| "A base é ruim" | 109K lances, rótulos Stockfish depth 15 confiáveis |

#### Causa raiz: features não separam as classes

As features posicionais descrevem o **ambiente** (material, mobilidade, segurança do rei), mas os erros de jogadores 1400–1700 são causados por **ameaças táticas concretas**: peças indefesas, cravadas, garfos — nenhuma capturada pelas 33 features V1.

In [ ]:
df_v1_feat = pd.read_csv(DATA_DIR / "features" / "features.csv")
v1_feature_cols = [c for c in df_v1_feat.columns if c != "label"]
y_binary = (df_v1_feat["label"] == "ruim").astype(int)

from scipy.stats import pointbiserialr

corrs = []
for col in v1_feature_cols:
    r, _ = pointbiserialr(y_binary, df_v1_feat[col])
    mean_bom = df_v1_feat.loc[y_binary == 0, col].mean()
    mean_ruim = df_v1_feat.loc[y_binary == 1, col].mean()
    std_pool = np.sqrt(
        (df_v1_feat.loc[y_binary == 0, col].std() ** 2
         + df_v1_feat.loc[y_binary == 1, col].std() ** 2) / 2
    )
    d = (mean_ruim - mean_bom) / std_pool if std_pool > 0 else 0
    corrs.append({"feature": col, "r": r, "cohen_d": abs(d)})

df_corr = pd.DataFrame(corrs).sort_values("cohen_d", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
top_corr = df_corr.head(10)
labels_pt = [FEATURE_TRANSLATION.get(f, f) for f in top_corr["feature"]]
colors_corr = ["#C44E52" if abs(r) >= 0.08 else "#8C8C8C" for r in top_corr["r"]]
ax.barh(range(len(top_corr)), top_corr["r"].abs().values, color=colors_corr)
ax.set_yticks(range(len(top_corr)))
ax.set_yticklabels(labels_pt)
ax.invert_yaxis()
ax.set_xlabel("|Correlação point-biserial|")
ax.set_title("Top 10 Features V1 — Correlação com Label", fontsize=12)
ax.axvline(0.10, color="green", linestyle="--", alpha=0.6, label="|r| = 0.10 (limiar útil)")
ax.axvline(0.20, color="red", linestyle="--", alpha=0.6, label="|r| = 0.20 (limiar bom)")
ax.legend(fontsize=8)

ax = axes[1]
colors_d = ["#DD8452" if d >= 0.20 else "#8C8C8C" for d in top_corr["cohen_d"]]
ax.barh(range(len(top_corr)), top_corr["cohen_d"].values, color=colors_d)
ax.set_yticks(range(len(top_corr)))
ax.set_yticklabels(labels_pt)
ax.invert_yaxis()
ax.set_xlabel("Cohen's d (separabilidade)")
ax.set_title("Top 10 Features V1 — Separabilidade entre Classes", fontsize=12)
ax.axvline(0.20, color="green", linestyle="--", alpha=0.6, label="d = 0.20 (pequeno)")
ax.axvline(0.50, color="red", linestyle="--", alpha=0.6, label="d = 0.50 (médio)")
ax.legend(fontsize=8)

plt.suptitle("Diagnóstico V1: As features não separam as classes", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Melhor feature V1: {top_corr.iloc[0]['feature']} → |r| = {top_corr.iloc[0]['r']:.3f}, "
      f"Cohen's d = {top_corr.iloc[0]['cohen_d']:.3f}")
print(f"Nenhuma feature atinge |r| ≥ 0.10 — poder preditivo individual negligível.")

### 6.7 Comparação V1 vs V2 — Impacto das Features Táticas

A V2 adicionou **19 features táticas** (peças indefesas, capturas com ganho, cravadas, rei avançado, tensão) para atacar diretamente a causa raiz identificada no diagnóstico. Esta seção compara os resultados lado a lado.

> **Nota:** Esta seção requer `VERSION ≥ 2` e os artefatos de comparação em `data/evaluation_v2/`.

In [ ]:
if VERSION >= 2:
    df_deltas_v2 = pd.read_csv(DATA_DIR / "evaluation_v2" / "v1_vs_v2_deltas.csv")
    df_v1v2 = pd.read_csv(DATA_DIR / "evaluation_v2" / "v1_vs_v2_metrics.csv")

    print(f"{'='*70}")
    print(f"  COMPARAÇÃO V1 → V2 — Deltas por Métrica")
    print(f"{'='*70}")
    for algo in ["Decision Tree", "Random Forest"]:
        subset = df_deltas_v2[df_deltas_v2["algo"] == algo]
        print(f"\n  {algo}:")
        for _, row in subset.iterrows():
            arrow = "↑" if row["delta"] > 0 else "↓"
            print(f"    {row['metric']:18s}  {row['v1']:.4f} → {row['v2']:.4f}  "
                  f"({arrow} {abs(row['delta_pp']):+.2f} pp)")

    metric_labels = {
        "accuracy": "Accuracy", "f1_ruim": "F1 (ruim)",
        "recall_ruim": "Recall (ruim)", "precision_ruim": "Precision (ruim)",
        "roc_auc": "ROC-AUC",
    }
    metrics_to_plot = ["accuracy", "f1_ruim", "recall_ruim", "precision_ruim", "roc_auc"]
    x = np.arange(len(metrics_to_plot))
    width = 0.20

    fig, ax = plt.subplots(figsize=(14, 5.5))
    bar_configs = [
        ("Decision Tree", "V1", "#A8C8E8", "DT V1"),
        ("Decision Tree", "V2", "#4C72B0", "DT V2"),
        ("Random Forest", "V1", "#F5C9A0", "RF V1"),
        ("Random Forest", "V2", "#DD8452", "RF V2"),
    ]
    for i, (algo, ver, color, label) in enumerate(bar_configs):
        row = df_v1v2[(df_v1v2["algo"] == algo) & (df_v1v2["version"] == ver)].iloc[0]
        values = [row[m] for m in metrics_to_plot]
        bars = ax.bar(x + i * width, values, width, label=label, color=color,
                      edgecolor="white", linewidth=0.5)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=7.5)

    ax.set_xticks(x + 1.5 * width)
    ax.set_xticklabels([metric_labels[m] for m in metrics_to_plot])
    ax.set_ylim(0, 0.85)
    ax.set_ylabel("Score")
    ax.set_title("Comparação V1 vs V2 — Todas as Métricas", fontsize=14)
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("⏭️  Comparação V1 vs V2 disponível apenas com VERSION ≥ 2.")

In [ ]:
if VERSION >= 2:
    dt_v1 = joblib.load(DATA_DIR / "models" / "decision_tree.joblib")
    rf_v1 = joblib.load(DATA_DIR / "models" / "random_forest.joblib")
    with open(DATA_DIR / "models" / "feature_names.json") as f:
        v1_feature_names = json.load(f)

    dt_v2 = joblib.load(DATA_DIR / "models_v2" / "decision_tree.joblib")
    rf_v2 = joblib.load(DATA_DIR / "models_v2" / "random_forest.joblib")
    with open(DATA_DIR / "models_v2" / "feature_names.json") as f:
        v2_feature_names = json.load(f)

    X_test_v1 = X_test[v1_feature_names]
    X_test_v2 = X_test[v2_feature_names]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    RocCurveDisplay.from_estimator(dt_v1, X_test_v1, y_test, ax=ax,
                                   name="DT V1", linestyle="--", alpha=0.6)
    RocCurveDisplay.from_estimator(dt_v2, X_test_v2, y_test, ax=ax,
                                   name="DT V2")
    RocCurveDisplay.from_estimator(rf_v1, X_test_v1, y_test, ax=ax,
                                   name="RF V1", linestyle="--", alpha=0.6)
    RocCurveDisplay.from_estimator(rf_v2, X_test_v2, y_test, ax=ax,
                                   name="RF V2")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set_title("Curvas ROC — V1 vs V2", fontsize=13)
    ax.legend(loc="lower right")

    ax = axes[1]
    PrecisionRecallDisplay.from_estimator(dt_v1, X_test_v1, y_test, ax=ax,
                                          name="DT V1", linestyle="--", alpha=0.6)
    PrecisionRecallDisplay.from_estimator(dt_v2, X_test_v2, y_test, ax=ax,
                                          name="DT V2")
    PrecisionRecallDisplay.from_estimator(rf_v1, X_test_v1, y_test, ax=ax,
                                          name="RF V1", linestyle="--", alpha=0.6)
    PrecisionRecallDisplay.from_estimator(rf_v2, X_test_v2, y_test, ax=ax,
                                          name="RF V2")
    prevalence = y_test.mean()
    ax.axhline(y=prevalence, color="k", linestyle="--", alpha=0.3,
               label=f"Prevalência ({prevalence:.3f})")
    ax.set_title("Curvas Precision-Recall — V1 vs V2", fontsize=13)
    ax.legend(loc="upper right", fontsize=8)

    plt.tight_layout()
    plt.show()
else:
    print("⏭️  Overlay de curvas V1 vs V2 disponível apenas com VERSION ≥ 2.")

In [ ]:
if VERSION >= 2:
    df_tact = pd.read_csv(DATA_DIR / "evaluation_v2" / "tactical_features_analysis.csv")

    fig, ax = plt.subplots(figsize=(10, 6))
    colors_tact = ["#C44E52" if imp >= 0.03 else "#DD8452" if imp >= 0.01 else "#8C8C8C"
                   for imp in df_tact["importance_rf_v2"]]
    ax.barh(range(len(df_tact)), df_tact["importance_rf_v2"].values, color=colors_tact)
    ax.set_yticks(range(len(df_tact)))
    ax.set_yticklabels(df_tact["feature_pt"])
    ax.invert_yaxis()
    ax.set_xlabel("Importância (Gini) — Random Forest V2")
    ax.set_title("Importância das 19 Features Táticas (V2)", fontsize=13)

    for i, (_, row) in enumerate(df_tact.iterrows()):
        ax.text(row["importance_rf_v2"] + 0.001, i, f"{row['importance_rf_v2']:.4f}",
                va="center", fontsize=8)

    plt.tight_layout()
    plt.show()

    print(f"Feature tática #1: {df_tact.iloc[0]['feature_pt']} "
          f"(importância = {df_tact.iloc[0]['importance_rf_v2']:.4f})")
    print(f"  → É também a feature #1 GERAL do RF V2, superando todas as 33 features V1.")
    print(f"\nDas 19 features táticas:")
    n_top15 = sum(1 for _, r in df_tact.iterrows() if r["importance_rf_v2"] >= 0.01)
    print(f"  {n_top15} têm importância ≥ 0.01 (relevantes)")
    print(f"  {len(df_tact) - n_top15} têm importância < 0.01 (contribuição marginal)")
else:
    print("⏭️  Análise de features táticas disponível apenas com VERSION ≥ 2.")

#### Síntese da comparação V1 → V2

| Aspecto | V1 (33 features) | V2 (52 features) | Variação |
|---------|-------------------|-------------------|----------|
| **RF F1-ruim** | 0.3525 | 0.3664 | +1.39 pp |
| **RF Recall-ruim** | 0.5523 | 0.5710 | +1.87 pp |
| **RF ROC-AUC** | 0.6837 | 0.7079 | +2.42 pp |
| **DT F1-ruim** | 0.3280 | 0.3429 | +1.49 pp |
| **DT ROC-AUC** | 0.6487 | 0.6744 | +2.57 pp |
| **Feature #1** | move_number | contested_squares | Nova feature tática lidera |

**Interpretação:**
- A melhoria é **consistente** em todas as métricas e ambos os modelos.
- O ganho é **modesto** (~1.4 pp em F1, ~2.4 pp em AUC) — confirma que features táticas leves ajudam, mas **não resolvem** o gap fundamental.
- `contested_squares` (casas disputadas por ambos os lados) é a feature mais forte do modelo inteiro, superando as 33 features V1.
- O teto de performance com features estáticas parece estar em torno de F1-ruim ~0.40 — para ir além, seriam necessárias features baseadas em análise de variantes (ou representações aprendidas).

### 6.8 Diagnóstico V2: Por que +1.4pp não basta?

A V2 adicionou features que detectam **perigos no tabuleiro** (peças indefesas, cravadas, tensão). A melhoria foi consistente mas modesta (+1.4pp em F1). O diagnóstico revela por quê:

> **Dois lances diferentes na mesma posição teriam exatamente as mesmas 52 features.**

Se a posição tem 2 peças penduradas e 1 cravada, essas features são **iguais** quer o jogador jogue `Nf3` (salvando a peça) ou `a3` (ignorando tudo). As features V1+V2 capturam o **cenário antes do lance**, mas não dizem se o lance **lida com os perigos ou os ignora**.

#### Evolução do tipo de informação

| Versão | O que sabe | O que falta |
|--------|-----------|-------------|
| V1 | "A posição tem material desigual, baixa mobilidade" | Não sabe que há ameaças |
| V2 | "Há 2 peças indefesas e 1 cravada" | Não sabe se o lance resolve ou ignora isso |
| **V3** | **"O lance deixou 1 peça indefesa e permite captura grátis"** | — |

**Conclusão:** precisamos de features que avaliem o **impacto do lance** — deltas antes/depois, respostas do adversário, e resultado de trocas de peças (SEE).

In [ ]:
if VERSION >= 2:
    df_v2_feat = pd.read_csv(DATA_DIR / "features" / "features_v2.csv")
    v2_feat_cols = [c for c in df_v2_feat.columns if c != "label"]
    y_v2_binary = (df_v2_feat["label"] == "ruim").astype(int)

    from scipy.stats import pointbiserialr

    corrs_v2 = []
    for col in v2_feat_cols:
        r, _ = pointbiserialr(y_v2_binary, df_v2_feat[col])
        mean_bom = df_v2_feat.loc[y_v2_binary == 0, col].mean()
        mean_ruim = df_v2_feat.loc[y_v2_binary == 1, col].mean()
        std_pool = np.sqrt(
            (df_v2_feat.loc[y_v2_binary == 0, col].std() ** 2
             + df_v2_feat.loc[y_v2_binary == 1, col].std() ** 2) / 2
        )
        d = (mean_ruim - mean_bom) / std_pool if std_pool > 0 else 0
        corrs_v2.append({"feature": col, "r": r, "cohen_d": abs(d)})

    df_corr_v2 = pd.DataFrame(corrs_v2).sort_values("cohen_d", ascending=False)
    top_v2 = df_corr_v2.head(10)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    labels_pt = [FEATURE_TRANSLATION.get(f, f) for f in top_v2["feature"]]
    colors_r = ["#C44E52" if abs(r) >= 0.10 else "#8C8C8C" for r in top_v2["r"]]
    ax.barh(range(len(top_v2)), top_v2["r"].abs().values, color=colors_r)
    ax.set_yticks(range(len(top_v2)))
    ax.set_yticklabels(labels_pt)
    ax.invert_yaxis()
    ax.set_xlabel("|Correlação point-biserial|")
    ax.set_title("Top 10 Features V2 — Correlação com Label", fontsize=12)
    ax.axvline(0.10, color="green", linestyle="--", alpha=0.6, label="|r| = 0.10")
    ax.axvline(0.20, color="red", linestyle="--", alpha=0.6, label="|r| = 0.20")
    ax.legend(fontsize=8)

    ax = axes[1]
    colors_d = ["#DD8452" if d >= 0.20 else "#8C8C8C" for d in top_v2["cohen_d"]]
    ax.barh(range(len(top_v2)), top_v2["cohen_d"].values, color=colors_d)
    ax.set_yticks(range(len(top_v2)))
    ax.set_yticklabels(labels_pt)
    ax.invert_yaxis()
    ax.set_xlabel("Cohen's d (separabilidade)")
    ax.set_title("Top 10 Features V2 — Separabilidade entre Classes", fontsize=12)
    ax.axvline(0.20, color="green", linestyle="--", alpha=0.6, label="d = 0.20 (pequeno)")
    ax.axvline(0.50, color="red", linestyle="--", alpha=0.6, label="d = 0.50 (médio)")
    ax.legend(fontsize=8)

    plt.suptitle("Diagnóstico V2: Melhoria real mas teto de features estáticas",
                 fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

    print(f"Melhor feature V2: {top_v2.iloc[0]['feature']} → |r| = {abs(top_v2.iloc[0]['r']):.3f}, "
          f"Cohen's d = {top_v2.iloc[0]['cohen_d']:.3f}")
    n_above = sum(1 for _, row in df_corr_v2.iterrows() if abs(row["r"]) >= 0.10)
    print(f"Features com |r| ≥ 0.10: {n_above} (vs 0 na V1)")
    print(f"\nMas o problema persiste: features pré-lance não distinguem lances "
          f"diferentes na mesma posição.")
else:
    print("⏭️  Diagnóstico V2 disponível apenas com VERSION ≥ 2.")

### 6.9 Comparação V1 → V2 → V3 — Impacto do Look-ahead

A V3 adicionou **15 features de look-ahead** em 3 grupos:
- **Deltas (G13):** diferença de peças indefesas, mobilidade, tensão e ameaças antes→depois do lance
- **Resposta do adversário (G14):** melhor captura disponível, possibilidade de xeque, capturas boas
- **SEE (G15):** Static Exchange Evaluation — resultado de sequências de captura num quadrado

Pela primeira vez, o modelo pode distinguir se o lance **resolve ou ignora** os perigos da posição.

> **Nota:** Esta seção requer `VERSION = 3` e os artefatos em `data/evaluation_v3/`.

In [ ]:
if VERSION >= 3:
    df_deltas_v3 = pd.read_csv(DATA_DIR / "evaluation_v3" / "v1_v2_v3_deltas.csv")
    df_v123 = pd.read_csv(DATA_DIR / "evaluation_v3" / "v1_v2_v3_metrics.csv")

    print(f"{'='*70}")
    print(f"  COMPARAÇÃO V1 → V2 → V3 — Deltas por Métrica")
    print(f"{'='*70}")
    for algo in ["Decision Tree", "Random Forest"]:
        subset = df_deltas_v3[df_deltas_v3["algo"] == algo]
        print(f"\n  {algo}:")
        for _, row in subset.iterrows():
            a12 = "↑" if row["delta_v2_v1"] > 0 else "↓"
            a23 = "↑" if row["delta_v3_v2"] > 0 else "↓"
            a13 = "↑" if row["delta_v3_v1"] > 0 else "↓"
            print(f"    {row['metric']:18s}  V1={row['v1']:.4f} → V2={row['v2']:.4f} → V3={row['v3']:.4f}  "
                  f"(total {a13} {abs(row['delta_v3_v1_pp']):.2f} pp)")

    metric_labels = {
        "accuracy": "Accuracy", "f1_ruim": "F1 (ruim)",
        "recall_ruim": "Recall (ruim)", "precision_ruim": "Precision (ruim)",
        "roc_auc": "ROC-AUC",
    }
    metrics_to_plot = ["accuracy", "f1_ruim", "recall_ruim", "precision_ruim", "roc_auc"]
    x = np.arange(len(metrics_to_plot))
    width = 0.13

    fig, ax = plt.subplots(figsize=(16, 6))
    bar_configs = [
        ("Decision Tree", "V1", "#C8D8E8", "DT V1"),
        ("Decision Tree", "V2", "#7BA3CC", "DT V2"),
        ("Decision Tree", "V3", "#4C72B0", "DT V3"),
        ("Random Forest", "V1", "#F5D8B0", "RF V1"),
        ("Random Forest", "V2", "#E8A86E", "RF V2"),
        ("Random Forest", "V3", "#DD8452", "RF V3"),
    ]
    for i, (algo, ver, color, label) in enumerate(bar_configs):
        row = df_v123[(df_v123["algo"] == algo) & (df_v123["version"] == ver)].iloc[0]
        values = [row[m] for m in metrics_to_plot]
        bars = ax.bar(x + i * width, values, width, label=label, color=color,
                      edgecolor="white", linewidth=0.5)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.006,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=6.5,
                    rotation=90 if len(bar_configs) > 4 else 0)

    ax.set_xticks(x + 2.5 * width)
    ax.set_xticklabels([metric_labels[m] for m in metrics_to_plot])
    ax.set_ylim(0, 0.95)
    ax.set_ylabel("Score")
    ax.set_title("Evolução V1 → V2 → V3 — Todas as Métricas", fontsize=14)
    ax.legend(loc="upper right", fontsize=8, ncol=2)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("⏭️  Comparação V1/V2/V3 disponível apenas com VERSION = 3.")

In [ ]:
if VERSION >= 3:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    RocCurveDisplay.from_estimator(dt_v1, X_test_v1, y_test, ax=ax,
                                   name="DT V1", linestyle=":", alpha=0.4)
    RocCurveDisplay.from_estimator(dt_v2, X_test_v2, y_test, ax=ax,
                                   name="DT V2", linestyle="--", alpha=0.6)
    RocCurveDisplay.from_estimator(dt, X_test, y_test, ax=ax,
                                   name="DT V3", linewidth=2)
    RocCurveDisplay.from_estimator(rf_v1, X_test_v1, y_test, ax=ax,
                                   name="RF V1", linestyle=":", alpha=0.4)
    RocCurveDisplay.from_estimator(rf_v2, X_test_v2, y_test, ax=ax,
                                   name="RF V2", linestyle="--", alpha=0.6)
    RocCurveDisplay.from_estimator(rf, X_test, y_test, ax=ax,
                                   name="RF V3", linewidth=2)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set_title("Curvas ROC — V1 vs V2 vs V3", fontsize=13)
    ax.legend(loc="lower right", fontsize=8)

    ax = axes[1]
    PrecisionRecallDisplay.from_estimator(dt_v1, X_test_v1, y_test, ax=ax,
                                          name="DT V1", linestyle=":", alpha=0.4)
    PrecisionRecallDisplay.from_estimator(dt_v2, X_test_v2, y_test, ax=ax,
                                          name="DT V2", linestyle="--", alpha=0.6)
    PrecisionRecallDisplay.from_estimator(dt, X_test, y_test, ax=ax,
                                          name="DT V3", linewidth=2)
    PrecisionRecallDisplay.from_estimator(rf_v1, X_test_v1, y_test, ax=ax,
                                          name="RF V1", linestyle=":", alpha=0.4)
    PrecisionRecallDisplay.from_estimator(rf_v2, X_test_v2, y_test, ax=ax,
                                          name="RF V2", linestyle="--", alpha=0.6)
    PrecisionRecallDisplay.from_estimator(rf, X_test, y_test, ax=ax,
                                          name="RF V3", linewidth=2)
    prevalence = y_test.mean()
    ax.axhline(y=prevalence, color="k", linestyle="--", alpha=0.3,
               label=f"Prevalência ({prevalence:.3f})")
    ax.set_title("Curvas Precision-Recall — V1 vs V2 vs V3", fontsize=13)
    ax.legend(loc="upper right", fontsize=7)

    plt.tight_layout()
    plt.show()
else:
    print("⏭️  Overlay de curvas V1/V2/V3 disponível apenas com VERSION = 3.")

In [ ]:
if VERSION >= 3:
    df_look = pd.read_csv(DATA_DIR / "evaluation_v3" / "lookahead_features_analysis.csv")

    v3_new_features = [
        "delta_hanging_player", "delta_hanging_opponent", "delta_hanging_value_player",
        "delta_threats_against_player", "delta_mobility_player", "delta_mobility_opponent",
        "delta_contested_squares", "delta_king_attackers_player",
        "opponent_best_capture_value", "opponent_can_check", "opponent_num_good_captures",
        "created_hanging_self", "see_of_move", "worst_see_against_player", "is_losing_capture",
    ]

    color_map = {}
    for f in feature_names:
        if f in v3_new_features:
            color_map[f] = "#55A868"
        elif f in v2_feature_names and f not in v1_feature_names:
            color_map[f] = "#C44E52"
        else:
            color_map[f] = "#4C72B0"

    imp = rf.feature_importances_
    idx = np.argsort(imp)[::-1][:20]

    fig, ax = plt.subplots(figsize=(10, 7))
    labels = [FEATURE_TRANSLATION.get(feature_names[i], feature_names[i]) for i in idx]
    values = imp[idx]
    colors = [color_map[feature_names[i]] for i in idx]
    bars = ax.barh(range(len(idx)), values, color=colors)
    ax.set_yticks(range(len(idx)))
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel("Importância (Gini)")
    ax.set_title("Top 20 Features — Random Forest V3", fontsize=13)
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                f"{val:.4f}", va="center", fontsize=8)

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="#4C72B0", label="V1 — Posicionais"),
        Patch(facecolor="#C44E52", label="V2 — Táticas"),
        Patch(facecolor="#55A868", label="V3 — Look-ahead"),
    ]
    ax.legend(handles=legend_elements, loc="lower right", fontsize=9)

    plt.tight_layout()
    plt.show()

    print("Look-ahead features no top 10 do RF V3:")
    for rank, i in enumerate(idx[:10], 1):
        f = feature_names[i]
        origin = "V3" if f in v3_new_features else ("V2" if f in v2_feature_names and f not in v1_feature_names else "V1")
        print(f"  #{rank}: {FEATURE_TRANSLATION.get(f, f):30s} ({origin}) — {imp[i]:.4f}")
else:
    print("⏭️  Análise de features look-ahead disponível apenas com VERSION = 3.")

#### Síntese da evolução V1 → V2 → V3

| Métrica (RF) | V1 | V2 | V3 | Δ total |
|-------------|-----|-----|-----|---------|
| **Accuracy** | 0.6827 | 0.6912 | 0.7849 | **+10.22 pp** |
| **F1 (ruim)** | 0.3525 | 0.3664 | 0.4312 | **+7.87 pp** |
| **Precision (ruim)** | 0.2589 | 0.2698 | 0.3676 | **+10.87 pp** |
| **Recall (ruim)** | 0.5523 | 0.5710 | 0.5215 | −3.08 pp |
| **ROC-AUC** | 0.6837 | 0.7079 | 0.7678 | **+8.41 pp** |
| **Feature #1** | move_number | contested_squares | worst_see_against_player | — |

**Interpretação:**
- O salto V2→V3 é o **maior de todo o projeto**: +9.37pp em accuracy, +6.48pp em F1-ruim, +5.99pp em AUC.
- A **meta de AUC ≥ 0.75 foi atingida** (0.7678). A meta de F1 ≥ 0.50 não foi atingida (0.4312), mas o salto é muito significativo.
- O recall da classe ruim **caiu** 4.95pp (V2→V3): as árvores mais profundas com features mais informativas fazem um **trade-off precision↑/recall↓**. Como a precision subiu 9.78pp, o F1 sobe porque o equilíbrio é mais favorável.
- Look-ahead features dominam o top 5: `worst_see_against_player` (#1), `delta_mobility_opponent` (#2), `opponent_best_capture_value` (#3). Pela primeira vez, o modelo "entende" o impacto do lance.
- O `max_depth` do RF subiu de 10→15 — features mais informativas permitem árvores mais profundas e expressivas.

---

## 7. Interpretação e Exemplos

### 7.1 Regras da Árvore de Decisão

A grande vantagem da Decision Tree é a **interpretabilidade**: as regras de decisão podem ser traduzidas diretamente para conceitos de xadrez.

In [ ]:
translated_names = [FEATURE_TRANSLATION.get(f, f) for f in feature_names]

fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    dt, feature_names=translated_names,
    class_names=["Bom", "Ruim"],
    filled=True, rounded=True,
    max_depth=3, fontsize=9, ax=ax,
    impurity=False, proportion=True,
)
ax.set_title("Árvore de Decisão (primeiros 3 níveis)", fontsize=15, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
_rules_suffix = {1: "", 2: "_v2", 3: "_v3"}
_rules_file = EVAL_DIR / f"decision_tree_rules_chess{_rules_suffix[VERSION]}.txt"
if _rules_file.exists():
    rules_text = _rules_file.read_text()
else:
    rules_text = Path("data/evaluation/decision_tree_rules_chess.txt").read_text()

print(rules_text[:3000])
if len(rules_text) > 3000:
    print("\n... (regras continuam)")

### 7.2 Interpretação das regras

As regras revelam padrões consistentes com a teoria de xadrez. Na V3, as features de look-ahead dominam as primeiras decisões:

| Regra aprendida (V3) | Interpretação no domínio |
|----------------------|-------------------------|
| **1ª decisão: pior SEE ≤ -0.50** | O lance deixa o jogador vulnerável a trocas desfavoráveis? Primeira pergunta é sobre consequências. |
| **É captura + SEE negativo** | Captura onde o jogador perde material na sequência de recapturas → forte indicador de lance ruim. |
| **Melhor captura adversário > 2** | Se o adversário pode capturar uma peça de valor ≥ 2 (cavalo/bispo), o lance é suspeito. |
| **Adversário pode dar xeque** | Lance que permite xeque do adversário tende a ser problemático. |
| **Δ mobilidade jogador < -21** | Lance que reduz drasticamente as opções do jogador — sinal de auto-restrição. |
| **Criou peça indefesa** | O lance deixou uma peça sem defensores — erro tático claro. |

**Evolução das regras:** Na V1, a 1ª decisão era "é captura?"; na V3, é "qual o pior resultado de troca?" — o modelo evoluiu de descrever o tipo de lance para avaliar suas consequências.

### 7.3 Análise qualitativa de erros

Vamos examinar exemplos concretos onde o modelo erra, para entender as limitações.

In [ ]:
_err_suffix = {1: "", 2: "_v2", 3: "_v3"}
_fp_file = EVAL_DIR / f"error_analysis_fp{_err_suffix[VERSION]}.csv"
_fn_file = EVAL_DIR / f"error_analysis_fn{_err_suffix[VERSION]}.csv"
if not _fp_file.exists():
    _fp_file = DATA_DIR / "evaluation" / "error_analysis_fp.csv"
    _fn_file = DATA_DIR / "evaluation" / "error_analysis_fn.csv"
df_fp = pd.read_csv(_fp_file)
df_fn = pd.read_csv(_fn_file)

def show_examples(df, error_type, model_name, n=5):
    subset = df[df["model"] == model_name].head(n)
    tipo = "FALSOS POSITIVOS" if error_type == "FP" else "FALSOS NEGATIVOS"
    desc = ('modelo disse "ruim", lance é bom' if error_type == "FP"
            else 'modelo disse "bom", lance é ruim')

    print(f"\n{'='*65}")
    print(f"  {tipo} — {model_name}")
    print(f"  ({desc})")
    print(f"{'='*65}")

    for i, (_, row) in enumerate(subset.iterrows(), 1):
        print(f"\n  {error_type}-{i}: {row['move_san']} (lance #{int(row['move_number'])}, {row['color']})")
        print(f"    Delta: {row['delta_cp']} cp | Partida: {row['game_site']}")
        feats = str(row.get("top_features", "")).split("; ")
        translated_feats = [
            f"{FEATURE_TRANSLATION.get(f.split('=')[0], f.split('=')[0])}={f.split('=')[1]}"
            for f in feats if "=" in f
        ]
        print(f"    Features: {'; '.join(translated_feats)}")

_model_suffix = {1: "", 2: " V2", 3: " V3"}
_dt_name = f"Decision Tree{_model_suffix[VERSION]}"
_rf_name = f"Random Forest{_model_suffix[VERSION]}"
show_examples(df_fp, "FP", _dt_name)
show_examples(df_fn, "FN", _dt_name)
print()
show_examples(df_fp, "FP", _rf_name)
show_examples(df_fn, "FN", _rf_name)

### 7.4 Discussão dos erros

**Falsos positivos** (modelo diz "ruim", lance é bom):
- Na V3, muitos FPs envolvem lances onde o SEE indica perda mas a posição tem compensação posicional (atividade, iniciativa).
- Lances em posições com alta tensão (`contested_squares` alto) onde o lance é adequado mas o contexto parece perigoso.

**Falsos negativos** (modelo diz "bom", lance é ruim):
- **Erros táticos de múltiplos lances**: combinações de 2-3 lances (ex.: sacrifício seguido de mate) que o look-ahead de 1 ply não alcança.
- Erros posicionais subtis onde a avaliação piora progressivamente ao longo de vários lances.

**Evolução dos erros ao longo das versões:**

| Versão | Tipo de FN dominante | O que falta |
|--------|---------------------|-------------|
| V1 | Qualquer erro tático | Detecção de ameaças |
| V2 | Erros onde a posição tem perigos que o lance ignora | Avaliação do impacto do lance |
| V3 | Combinações multi-lance e erros posicionais profundos | Look-ahead de 2+ plies ou avaliação de engine |

**Limitação fundamental na V3**: o look-ahead de 1 ply (antes→depois do lance + resposta imediata do adversário) resolve ameaças simples mas não captura sacrifícios ou combinações. Para isso, seria necessário look-ahead de 2+ plies ou features baseadas em avaliação de engine — que introduzem risco de circularidade com o rótulo Stockfish.

---

## 8. Conclusões e Trabalhos Futuros

### Resultados principais

| Aspecto | V1 (33 features) | V2 (52 features) | V3 (67 features) |
|---------|-------------------|-------------------|-------------------|
| **Dataset** | 109,290 lances, 3,000 partidas | Mesmo dataset | Mesmo dataset |
| **RF Accuracy** | 0.6827 | 0.6912 (+0.85 pp) | **0.7849 (+10.22 pp)** |
| **RF F1-ruim** | 0.3525 | 0.3664 (+1.39 pp) | **0.4312 (+7.87 pp)** |
| **RF Precision-ruim** | 0.2589 | 0.2698 (+1.09 pp) | **0.3676 (+10.87 pp)** |
| **RF AUC** | 0.6837 | 0.7079 (+2.42 pp) | **0.7678 (+8.41 pp)** ✅ |
| **DT F1-ruim** | 0.3280 | 0.3429 (+1.49 pp) | **0.3821 (+5.41 pp)** |
| **Feature #1** | move_number | contested_squares | worst_see_against_player |
| **Interpretabilidade** | Regras sobre tipo/fase | + tensão tática | + consequências do lance |

### Evolução V1 → V2 → V3: o ciclo científico

O projeto seguiu um ciclo de **diagnóstico → melhoria** executado três vezes:

**V1 — O cenário (33 features posicionais)**
- Features que descrevem o *ambiente* da posição: material, mobilidade, segurança do rei, estrutura de peões.
- Resultado: RF F1-ruim = 0.35, AUC = 0.68.
- Diagnóstico: nenhuma feature com |r| ≥ 0.10. As features não capturam as **causas** dos erros.

**V2 — As ameaças (+ 19 features táticas)**
- Peças indefesas, capturas com ganho, cravadas, tensão posicional.
- Resultado: RF F1-ruim = 0.37 (+1.4 pp), AUC = 0.71 (+2.4 pp).
- Diagnóstico: `contested_squares` lidera (d = 0.40), mas dois lances na mesma posição têm **as mesmas features** — falta avaliar o **impacto do lance**.

**V3 — O impacto (+ 15 features look-ahead)**
- Deltas antes→depois, resposta do adversário, Static Exchange Evaluation.
- Resultado: **RF F1-ruim = 0.43 (+6.5 pp), AUC = 0.77 (+6.0 pp)** — o maior salto do projeto.
- `worst_see_against_player` é feature #1: pela primeira vez, o modelo avalia **consequências**.

### O que aprendemos

- **Tipo de informação > quantidade de features**: as 15 features V3 trouxeram +6.5pp em F1 vs +1.4pp das 19 features V2. A chave não é descrever mais o cenário, mas avaliar o **impacto do lance**.
- **SEE é a feature mais discriminativa**: o resultado de trocas de peças (técnica de 1980s) supera todas as features posicionais e táticas combinadas.
- **Trade-off precision/recall**: o RF V3 tem recall menor que o V2 (-4.95pp) mas precision muito maior (+9.78pp). Menos alarmes falsos = modelo mais útil na prática.
- **Meta AUC ≥ 0.75 atingida** (0.7678). Meta F1 ≥ 0.50 não atingida (0.4312), mas o salto é significativo — seria necessário look-ahead de 2+ plies ou features de engine para ir além.
- A Decision Tree V3 evoluiu: a 1ª decisão passou de "é captura?" (V1) para "qual o pior resultado de troca?" (V3) — interpretação mais rica e orientada a consequências.
- O ciclo **treino → diagnóstico → melhoria → re-treino** é o resultado central: cada iteração foi motivada por evidência quantitativa e resultou em melhoria mensurável.

### Limitações

- Dataset limitado a uma faixa de rating (1400–1700 Lichess) e período (jan/2015)
- Look-ahead de 1 ply não captura combinações de 2+ lances (sacrifícios, mating patterns)
- Classificação binária (bom/ruim) é simplista — zona cinzenta de 27K lances descartada
- Rótulos dependem do Stockfish (depth 15) — risco de circularidade se features usarem engine
- F1-ruim = 0.43 limita a utilidade prática (ainda ~57% de falsos alarmes)

### Trabalhos futuros

1. **Look-ahead de 2 plies**: simular lance + resposta + 2ª jogada para capturar combinações táticas
2. **Gradient boosting** (XGBoost/LightGBM) com as 67 features — potencial de +2-5pp sobre RF
3. **Classificação multiclasse**: brilhante / bom / imprecisão / erro / blunder (5 classes do Lichess)
4. **Modelos por fase**: separar abertura, meio-jogo e final — cada fase tem padrões de erro distintos
5. **Representações aprendidas**: embeddings de posições via CNNs ou transformers
6. **Aplicação prática**: módulo que sugere ao jogador revisar posições onde o modelo prevê erro alto